In [1]:
import pandas as pd
from PIL import Image
import io
import requests
from IPython.display import display, clear_output
import ipywidgets as widgets
import json

/Users/wpp74634/Desktop/data-assignment/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
gals_df = pd.read_csv('data/desert.csv')

In [3]:
print(gals_df.iloc[0]["location"])
#it works!

Glass Dunes


In [4]:
display(gals_df.head())

,location,discovery,category,time_of_day,description,story,asset
0,Glass Dunes,Desert glass,Geology,Day,A smooth fragment of glass lies among the sand.,Sand can change form under extraordinary heat....,desert_glass.png
1,Glass Dunes,Ripple marks,Geology,Dawn,Parallel ridges stretch across the dune surface.,The wind has been writing here all night. By m...,ripple_marks.png
2,Glass Dunes,Beetle tracks,Wildlife,Dawn,Tiny paired tracks cross an otherwise untouche...,Something small has crossed this enormous plac...,beetle_tracks.png
3,Glass Dunes,Buried seed,Botany,Dusk,A seed rests just beneath the dry surface.,It has not disappeared. It is waiting for the ...,buried_seed.png
4,Glass Dunes,Wind-polished stone,Geology,Day,One side of a small stone is worn smooth by bl...,Change can happen so slowly that no one notice...,polished_stone.png


In [5]:
#i need to filter and make a dataset because 2 things are super important in my game setup, location and time of day and then their relation to each other 
#for this assignment, Im only going to explore one location but full game play 

moon_hollow_df = gals_df[gals_df["location"] == "Moon Hollow"]
moon_hollow_df

,location,discovery,category,time_of_day,description,story,asset
12,Moon Hollow,Night-blooming flower,Botany,Night,A pale desert flower opens after sunset.,Not everything reveals itself in daylight.,night_flower.png
13,Moon Hollow,Moth visitor,Wildlife,Night,A moth hovers near a flower in the cool evenin...,"In the dark, a quiet exchange takes place betw...",moth.png
14,Moon Hollow,Cooled rock,Geology,Dusk,A stone that held the daytime heat now feels c...,"The desert is not one temperature, one mood, o...",cooled_rock.png
15,Moon Hollow,Moon shadow,Atmosphere,Night,A tall rock casts a long shadow under a bright...,A familiar landmark becomes unfamiliar when th...,moon_shadow.png
16,Moon Hollow,Burrowing entrance,Wildlife,Dusk,A small opening leads beneath the sand.,Aboveground is only part of the desert. Anothe...,burrow.png
17,Moon Hollow,Night sky,Atmosphere,Night,Stars crowd the sky above the hollow.,"Without the daylight, the desert reveals a dif...",night_sky.png


In [6]:
# i want to make a list for everything collected but i dont want to allow duplicates so im going to use a set, to start it's going to be empty
discovered = set()
# when the game first starts, the player hasn't encountered anything yet, so im setting the current discovery to none
current_discovery = None


In [7]:
#player chooses the time then starts the game 
time_selector = widgets.Dropdown(
    options=["Dawn", "Day", "Dusk", "Night"],
    value="Day",
    description="Time:"
)

In [8]:
# 3 buttons - to start the game, to explore the desert and to investigate an object --> invesigate an object is conditional though
start_button = widgets.Button(description="Start Game")
explore_button = widgets.Button(description="Explore")
investigate_button = widgets.Button(description="Investigate")

In [9]:
# making investigate button conditional (is there a better way to do this? like a state tracker) 
investigate_button.disabled = True


In [10]:
#game text area
output = widgets.Output()

# tracking the number of discoveries
progress = widgets.IntProgress(
    value=0,
    min=0,
    max=len(moon_hollow_df),
    description="Found:"
)

In [11]:
#i decided to break 
def gameControls(button): 
    print("DEBUG: Start button was clicked")
    with output:
            clear_output()
           
            print("Welcome to BUFFALO GALS DESERT")
            print()
            print("Welcome to Moon Hollow.")
            print()
            print("The desert appears empty, but not")
            print("everything reveals itself in daylight.")
            print()
            print("Change the time of day and explore.")
            print("Discover the things hidden in the desert.")

            # display(time_selector)
            # display(explore_button)
            # display(investigate_button)
            # display(progress)
    

In [12]:
#function for everytime the player wants to explore 
def explore(button):
    print("DEBUG: Explore function is running")

    global current_discovery #this is telling this function NOT to create a local vairiable with the same name as the global one we defined but to use the global one since it's also our tracker 

    # Get the value for the selected time
    selected_time = time_selector.value

    # get discoveries available at this time
    available = moon_hollow_df[
        moon_hollow_df["time_of_day"] == selected_time
    ]

    # filter out the discoveries already found
    remaining = []
    
    for index, row in available.iterrows():
        if row["discovery"] not in discovered:
            remaining.append(index)
    
    available = available.loc[remaining]

    with output:
        clear_output()

        print("MOON HOLLOW")
        print("Time:", selected_time)
        print() #newline --> is there an easier way to newline 

        
        #based on the dataset, check the discoveries, if none are available .empty will return ture 
        if available.empty:
            current_discovery = None
            investigate_button.disabled = True

            print("Nothing new reveals itself here.")
            print("Try another time of day.")

        else: #if there are discoveries at that time 
            # Choose a random discovery
            current_discovery = available.sample().iloc[0]

            print("You notice something unusual...")
            print()
            print("Clue:", current_discovery["category"])
            print()
            print("Click Investigate to learn more.")

           #set variable to false
            investigate_button.disabled = False
            

In [13]:
#when user click on an object and needs to investigate 
def investigate(button):

    global current_discovery #again, using and manipulating the same variable prev defined 
    
    #if they're not currently exploring anything just leave the function  
    if current_discovery is None:
        return

    # otherwise, get the name of the thing being discovered 
    discovery_name = current_discovery["discovery"]

    # count how many discoveries the player has collected
    # and update the progress bar with that number
    discovered.add(discovery_name)
    progress.value = len(discovered)

    with output:
        clear_output()
        #get relevent info about the object from the CSV such as description and story 
        print("NEW DISCOVERY!")
        print()
        print(discovery_name.upper())
        print()

        print(current_discovery["description"])
        print()
        print("--- THE STORY ---")
        print(current_discovery["story"])
        print()
        
       # i also want to show how many discoveries the user has collected and so len(discovered) counts the collected items and len(moon_hollow) counts the total discoveries available at this level
        print("Discovered:", len(discovered), "/", len(moon_hollow_df))

        if len(discovered) == len(moon_hollow_df): # check if the player has collected every discovery in moon hollow

            print()
            print("You've discovered everything in Moon Hollow!")

    # Reset until the player explores again
    current_discovery = None
    investigate_button.disabled = True

In [14]:
# this function runs whenever the player changes the time of day

def change_time(change):

    global current_discovery

    # reset the current discovery because the player has changed the time
    current_discovery = None
    investigate_button.disabled = True

    with output:
        clear_output()

        print("The desert shifts into", change["new"].lower() + ".")
        print()
        print("Click Explore to see what's here.")

In [15]:
#debugging code 
#gameControls(None)


In [16]:
# connect the start button to the gameControls function
start_button.on_click(gameControls)
explore_button.on_click(explore)

investigate_button.on_click(investigate)

time_selector.observe(change_time, names="value")

In [17]:
print("Welcome to BUFFALO GALS DESERT")
print("Chapter 1: Moon Hollow")
print()

display(start_button)
display(time_selector)
display(explore_button)
display(investigate_button)
display(progress)
display(output)

Welcome to BUFFALO GALS DESERT
Chapter 1: Moon Hollow



Button(description='Start Game', style=ButtonStyle())

Dropdown(description='Time:', index=1, options=('Dawn', 'Day', 'Dusk', 'Night'), value='Day')

Button(description='Explore', style=ButtonStyle())

Button(description='Investigate', disabled=True, style=ButtonStyle())

IntProgress(value=0, description='Found:', max=6)

Output()

In [18]:
# debugging code because nothing was printing 
#time_selector.value = "Night"

# explore(None)

# print("Current discovery:", current_discovery)

In [19]:
# debugging code to remove the old connection and connect the explore button to the function
#explore_button.on_click(explore, remove=True)
#explore_button.on_click(explore)

In [27]:
#debugging to see if jupyter is communicating with my python kernels, it is not :( 
test_slider = widgets.IntSlider(
    value=5,
    min=0,
    max=10,
    description="Test:"
)

display(test_slider)

IntSlider(value=5, description='Test:', max=10)

In [28]:
print(test_slider.value)

5
